In [1]:
#I valori di delta sono stati decisi osservando le oscillazioni della loss in un test
class EarlyStopping:
    def __init__(self, patience=5, delta=0.0, verbose=False):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.best_loss = None
        self.no_improvement_count = 0
        self.stop_training = False
        self.best_epoch=0
    
    def check_early_stop(self, val_loss,epoch):
        if self.best_loss is None or val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.no_improvement_count = 0
            self.best_epoch = epoch
        else:
            self.no_improvement_count += 1
            if self.no_improvement_count >= self.patience:
                self.stop_training = True
                if self.verbose:
                    print("Stopping early as no improvement has been observed.")

In [2]:
import torch.nn as nn
import torch.nn.functional as F



class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.kernel_size = kernel_size
        self.dilation = dilation
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, dilation=dilation)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None

    def forward(self, x):
        # Compute left padding so conv remains causal (no lookahead)
        pad = (self.kernel_size - 1) * self.dilation

        # First causal convolution (pad only on left)
        out = F.pad(x, (pad, 0))
        out = F.relu(self.conv1(out))

        # Second causal convolution
        out = F.pad(out, (pad, 0))
        out = self.conv2(out)

        # Add residual (identity) connection
        res = x if self.downsample is None else self.downsample(x)
        return F.relu(out + res)

class TCNLayer(nn.Module):
    def __init__(self, num_inputs, hidden_dimension, blocks, kernel_size):

        super().__init__()
        layers = []
        in_ch = num_inputs
        # Create a stack of ResidualBlocks with dilations 1, 2, 4...
        for i in range(blocks):
            dilation = 2 ** i      # La dilatazione consente di eseguire la convoluzione saltando i blocchi adiacenti al blocco attuale di d passi
            layers.append(ResidualBlock(in_ch, hidden_dimension, kernel_size, dilation))
            in_ch = hidden_dimension
        self.tcn = nn.Sequential(*layers)
        # Final linear layer to produce a single output value
        

    def forward(self, x):
        """
        x ha forma (batch_size, channels, seq_len).
        """
        return self.tcn(x)            # (batch, channels, seq_len)
              

In [3]:
import pandas as pd
import numpy as np

class Preprocessing:
    def __init__(self, data_path, adj_path):
        self.data_path = data_path
        self.adj_path = adj_path

    def normalization(self, use_graph=False):
        # Caricamento Dati e Indicizzazione Temporale
        df = pd.read_csv(self.data_path)
        threshold = 0.1
        if use_graph:
            if self.adj_path is None:
                raise ValueError("use_graph=True ma non è stato fornito adj_path.")
            adj = pd.read_pickle(self.adj_path)
            adj_matrix = adj[2].copy()
            adj_matrix[adj_matrix < threshold] = 0.0
        else:
            adj_matrix = None

        datetime_col = df.columns[0]
        df[datetime_col] = pd.to_datetime(df[datetime_col])
        df = df.set_index(datetime_col)

        sensor_cols = df.columns.tolist()

        # Distinzione Spaziale degli Zeri
        network_spatial_mean = df[sensor_cols].mean(axis=1)

        is_zero = (df[sensor_cols] == 0)
        is_global_blackout = (network_spatial_mean == 0)
        is_isolated_fault_mask = (network_spatial_mean > 10).values[:, None]
        is_fault = is_zero & (is_isolated_fault_mask | is_global_blackout.values[:, None])

        df_cleaned = df.copy()
        df_cleaned[is_fault] = np.nan

        # Creazione Metadati Temporali
        dow = df_cleaned.index.dayofweek
        time_slot = df_cleaned.index.hour * 12 + df_cleaned.index.minute // 5
        meta_df = pd.DataFrame({'dow': dow, 'time_slot': time_slot}, index=df_cleaned.index)

        train_ratio = 0.70
        train_end = int(len(df_cleaned) * train_ratio)

        df_train = df_cleaned.iloc[:train_end]
        meta_train = meta_df.iloc[:train_end]

        # Calcolo medie storiche SOLO sui dati di Training
        train_grouped = df_train.groupby([meta_train['dow'], meta_train['time_slot']]).mean()

        # Mappatura ed Imputazione sul Dataset Completo
        mapped_means = meta_df.merge(
            train_grouped,
            on=['dow', 'time_slot'],
            how='left'
        ).set_index(df_cleaned.index)[sensor_cols]

        df_imputed = df_cleaned.fillna(mapped_means)
        df = df_imputed.interpolate(method='time').ffill().bfill()

        print("Numero di NaN residui:", df.isna().sum().sum())

        # Splitting
        n = len(df)
        train_end = int(n * 0.70)
        val_end = int(n * 0.80)

        x_train = df.iloc[:train_end]
        x_val   = df.iloc[train_end:val_end]
        x_test  = df.iloc[val_end:]

        # Z-Score Globale su Train
        mean = x_train.values.mean()
        std  = x_train.values.std()
        x_train_norm = (x_train - mean) / std
        x_val_norm   = (x_val - mean) / std
        x_test_norm  = (x_test - mean) / std

        print(f"Global Mean (Train): {mean:.4f}")
        print(f"Global Std (Train):  {std:.4f}")

        # Normalizzazione delle medie storiche per la baseline HA
        mapped_means_norm = (mapped_means - mean) / std
        ha_val_norm  = mapped_means_norm.iloc[train_end:val_end]
        ha_test_norm = mapped_means_norm.iloc[val_end:]

        # Finestre Scorrevoli
        X_train, Y_train = self.sliding_windows(x_train_norm)
        X_val, Y_val     = self.sliding_windows(x_val_norm)
        X_test, Y_test   = self.sliding_windows(x_test_norm)

        _, Y_val_ha  = self.sliding_windows(ha_val_norm)
        _, Y_test_ha = self.sliding_windows(ha_test_norm)

        return X_train, Y_train, X_val, Y_val, X_test, Y_test, Y_val_ha, Y_test_ha, adj_matrix, train_grouped

    def sliding_windows(self, data, window_in=12, window_out=12):
        if hasattr(data, 'values'):
            data = data.values

        X, Y = [], []
        num_samples = len(data) - window_in - window_out + 1

        for i in range(num_samples):
            X.append(data[i: i + window_in])
            Y.append(data[i + window_in: i + window_in + window_out])

        return np.array(X), np.array(Y)

In [4]:
#Questo modello è senza grafo -> dalla traccia il nostro obiettivo è fare #
#un confronto equo tra i modelli
#questo modello utilizza ancora la TCN -> caso di ablazione quindi serve per forza

#Architettura modello: lascio la TCN identica, tolgo la GCN da quello di prima
#isolo totalmente il grafo
#ci aspettiamo risultati che dipendono esclusivamente dal passaggio di informazioni spaziale tra le strade e i sensori

#COME LAVORA??
#lavora sfruttando gli ultimi 12 passi temporali di ciascun sensore in modo totalmente isolato ed indipendente, senza sapere cosa
#sta succedendo nei sensori adiacenti o nelle strade vicine
#in sintesi: NO GRAPH = TCN PURA che lavora solo sull'asse dei tempo, sensore per sensore


import torch
import torch.nn as nn
import torch.nn.functional as F


class Temporal_Model(nn.Module):
    def __init__(self, input_dim=1, out_dim=3, hidden_dim=32, kernel_size=2, num_layers=3,num_blocks=3, dropout=0.3):
        super(Temporal_Model, self).__init__()

        self.num_layers=num_layers
        self.num_blocks=num_blocks
        self.dropout=dropout

        #Creazione strati TCN
        tcn_layers=[]
        for i in range(num_layers):
            in_ch=input_dim if i==0 else hidden_dim
            #l'ultimo strato mantiene hidden_dim per la testa di output
            out_ch=hidden_dim
            tcn_layers.append(TCNLayer(in_ch, out_ch, blocks=num_blocks, kernel_size=kernel_size))
        self.tcn_layers=nn.ModuleList(tcn_layers)
        self.drop=nn.Dropout(dropout)
        self.l1=nn.Linear(hidden_dim, out_dim) #mediana
        self.l2=nn.Linear(hidden_dim, out_dim) #quantile inferiore
        self.l3=nn.Linear(hidden_dim, out_dim) #quantile superiore

    def forward(self, x):
        #input x shape: (B, T, N, F) oppure (B, T, N)
        #output q10, q50, q90 di shape ciascuno (B, N, 3)
        if x.ndim==3:
            #vedo se manca la dimensione F, se manca la aggiungo
            x=x.unsqueeze(-1)
        B, T, N, F_in=x.shape

        #permutiamo e uniamo Batch e Nodi per la Conv1D
        #(B, N, T, F)-> (B, N, F, T) -> (B*N, F, T)
        x_tcn_in=x.permute(0, 2, 3, 1).reshape(B*N, F_in, T)

        #passaggio attraverso gli strati TCN
        for layer in self.tcn_layers:
            x_tcn_in=layer(x_tcn_in)
            x_tcn_in=self.drop(x_tcn_in)

        #estrazione dell'ultimo step temporale: shape (B*N, hidden_dim)
        out_last=x_tcn_in[:, :, -1]

        #ripristino formula (B, N, hidden_dim)
        out=out_last.reshape(B, N,-1 )

        #calcolo mediana e quantili
        median=self.l1(out)
        low_quantile=F.softplus(self.l2(out))
        high_quantile=F.softplus(self.l3(out))

        q10=median-low_quantile
        q50=median
        q90=median+high_quantile

        return q10, q50, q90

In [5]:
import torch 
import torch.nn as nn
import torch.nn.functional as F



class GCNLayer(nn.Module):

    def __init__(self, in_features, out_features, K, dropout):
        super(GCNLayer, self).__init__()
        self.K = K
        self.linears = nn.ModuleList(
            [nn.Linear(in_features, out_features) for _ in range(K + 1)]  # un peso per ordine
        )
        self.dropout = dropout

    def forward(self, x, T):
        out_list = []
        for k, T_k in enumerate(T):
            t_batched = T_k.unsqueeze(0).expand(x.size(0), -1, -1)   # (batch, N, N) Espando le dimensioni del vettore ricopiandolo lungo la grandezza del batch
            out = torch.bmm(t_batched, x)
            out = self.linears[k](out)                                
            out_list.append(out)
        out_sum=torch.sum(torch.stack(out_list, dim=0), dim=0)         # somma SOLO sui K+1 ordini
        out_sum=F.relu(out_sum)
        return  F.dropout(out_sum, p=self.dropout, training=self.training)


class GCN(nn.Module):
  
    
    def __init__(self, input_dim,out_dim, hidden_dim,T_list, kernel_size=3, num_layers=3,num_blocks=3, dropout=0.5,):
        super(GCN, self).__init__()
        
        # Store model hyperparameters.
        self.num_layers = num_layers
        self.dropout = dropout
        self.K = len(T_list)-1
        for k, T_k in enumerate(T_list):
            self.register_buffer(f"T_{k}", T_k)

        layers=[]
        for i in range(num_layers):
            if i== 0:
                layers.append(GCNLayer(input_dim, hidden_dim, self.K, dropout))
            elif i==num_layers-1 and num_layers > 1:
                layers.append(GCNLayer(hidden_dim, hidden_dim, self.K, dropout))
                
            else:
                layers.append(GCNLayer(hidden_dim, hidden_dim, self.K, dropout))

        self.gcn_layers= nn.ModuleList(
            [layers[i] for i in range(num_layers)]
        )
        layers=[]
        for i in range(num_layers):
            if i== num_layers-1:
                layers.append(TCNLayer(hidden_dim, out_dim, num_blocks, kernel_size))
            else:
                layers.append(TCNLayer(hidden_dim, hidden_dim, num_blocks, kernel_size))

        self.tcn_layers= nn.ModuleList(
            [layers[i] for i in range(num_layers)]
        )

        self.linear1=nn.Linear(out_dim, 3)
        self.linear2=nn.Linear(out_dim, 3)
        self.linear3=nn.Linear(out_dim, 3)
        
        
        
  
        
        

     
    def forward(self, x):
        # x: (B, T, N, F)
        B, T, N, _ = x.shape
        T_list = [getattr(self, f"T_{k}") for k in range(self.K + 1)]

        for i in range(self.num_layers):
        
            x_gcn_in = x .reshape(B * T, N, -1)   # (B*T, N, F)
            x_gcn_out = self.gcn_layers[i](x_gcn_in, T_list)          # (B*T, N, hidden_dim)
            F_hidden = x_gcn_out.shape[-1]
            x = x_gcn_out.reshape(B, T, N, F_hidden).permute(0, 2, 1, 3)  # torna a (B, N, T, hidden_dim)

            # --- TCN: fonde batch e N ---
            x_tcn_in = x.reshape(B * N, T, F_hidden).permute(0, 2, 1)     # (B*N, F, T)
            x_tcn_out = self.tcn_layers[i](x_tcn_in)                       # (B*N, F', T)
            F_out = x_tcn_out.shape[1]
            x = x_tcn_out.reshape(B, N, F_out,T ).permute(0, 3, 1,2)       # torna a (B, T,N, F')



        #Adesso genero i  quantili finali su 3/6/12 step temporali
        #eseguo la previsione unicamente sull'ultimo istante temporale generato (B, N, F')

        out=x[:, -1, :,:]

        median= self.linear1(out)
        #Gli altri quantili sono scritti solo in forma di incremento e decremento in modo che rimangano sopra/sotto il valore della mediana
        #softplus(x) = log(1 + e^x) consente di ottenere valori sempre positivi 
        low_quantile= F.softplus(self.linear2(out))
        high_quantile= F.softplus(self.linear3(out))    



        return median-low_quantile, median, median+high_quantile 



    


    




   

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GIN_Layer(nn.Module):

    def __init__(self, in_features, out_features, eps, dropout):
        super(GIN_Layer, self).__init__()
        self.eps=eps
        self.mlp=nn.Sequential(
            nn.Linear(in_features, out_features),
            nn.ReLU(),
            nn.Linear(out_features, out_features)
        )
        self.dropout = dropout

    def forward(self, x, A_norm):
        #A è la matrice di adiacenza con i self_loops
        A=A_norm.unsqueeze(0).expand(x.size(0),-1,-1)
        neighbor_Agg=torch.bmm(A, x)
        #formula_GIN
        out=self.mlp((1.0+self.eps)*x+neighbor_Agg)
        out=F.relu(out)
        return F.dropout(out, p=self.dropout, training=self.training)


class GIN(nn.Module):
    def __init__(self, input_dim, out_dim, hidden_dim, eps, A, kernel_size=3, num_layers=3, num_blocks=3, dropout=0.3):
        super(GIN, self).__init__()

        self.num_layers = num_layers
        self.dropout = dropout
        self.eps = eps

        # Adiacenza normalizzata
        self.register_buffer("A_norm", A)

        # Layer GIN
        gin_layers = []
        for i in range(num_layers):
            in_f = input_dim if i == 0 else hidden_dim
            gin_layers.append(GIN_Layer(in_f, hidden_dim, dropout=dropout, eps=eps))
        self.gin_layers = nn.ModuleList(gin_layers)

        # Strati TCN Temporali (Identici a GCN/Temporal_Model)
        tcn_layers = []
        for i in range(num_layers):
            out_f = out_dim if i == num_layers - 1 else hidden_dim
            tcn_layers.append(TCNLayer(hidden_dim, out_f, num_blocks, kernel_size))
        self.tcn_layers = nn.ModuleList(tcn_layers)

        # Teste Quantiliche
        self.linear1 = nn.Linear(out_dim, 3)
        self.linear2 = nn.Linear(out_dim, 3)
        self.linear3 = nn.Linear(out_dim, 3)

    def forward(self, x):
        # x: (B, T, N, F)
        B, T, N, _ = x.shape

        for i in range(self.num_layers):

            x_gin_in = x.reshape(B * T, N, -1)
            x_gin_out = self.gin_layers[i](x_gin_in, self.A_norm)
            F_hidden = x_gin_out.shape[-1]
            x = x_gin_out.reshape(B, T, N, F_hidden).permute(0, 2, 1, 3)

            x_tcn_in = x.reshape(B * N, T, F_hidden).permute(0, 2, 1)
            x_tcn_out = self.tcn_layers[i](x_tcn_in)
            F_out = x_tcn_out.shape[1]
            x = x_tcn_out.reshape(B, N, F_out, T).permute(0, 3, 1, 2)

        out = x[:, -1, :, :]

        median = self.linear1(out)
        low_quantile = F.softplus(self.linear2(out))
        high_quantile = F.softplus(self.linear3(out))

        return median - low_quantile, median, median + high_quantile

In [7]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from itertools import product


#Riproducibilità
torch.manual_seed(67)
np.random.seed(67)







def make_loaders(train_dataset, val_dataset, test_dataset, batch_size=32):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


def chebyshev_pol(L, K,device):
    L_shape = L.shape[0]
    T_0 = torch.eye(L_shape, dtype=L.dtype).to(device)
    T_1 = L
    T = [T_0, T_1]
    for i in range(2, K + 1):
        T_i = 2 * L @ T[i - 1] - T[i - 2]
        T.append(T_i)
    return T


def graph_to_matrices(adj_matrix):
    n_nodes = adj_matrix.shape[0]
    A_tilde = adj_matrix.astype(np.float32)
    A_sim = (A_tilde + A_tilde.T) / 2
    D = np.diag(np.sum(A_sim, axis=1))
    D_inv_sqrt = np.diag(np.power(np.diag(D), -0.5, where=np.diag(D) > 0))

    A_norm = D_inv_sqrt @ A_sim @ D_inv_sqrt
    L = np.eye(n_nodes) - A_norm
    eigenvalues = np.linalg.eigvalsh(L)
    max_eig = eigenvalues[-1]
    L_norm = 2 * L / max_eig - np.eye(n_nodes)

    return torch.Tensor(L_norm).float(), torch.Tensor(A_norm).float()


def pinball_loss(quantiles, y, y_pred):
    total_loss = 0
    for q, y_p in zip(quantiles, y_pred):
        loss = torch.max(q * (y - y_p), (1 - q) * (y_p - y))
        total_loss += loss.mean()
    return total_loss


def historical_average_baseline(y_true, y_pred_ha, quantiles=[0.1, 0.5, 0.9]):
    # Convertiamo in tensori PyTorch se sono array numpy
    if isinstance(y_true, np.ndarray):
        y_true = torch.from_numpy(y_true).float()
    if isinstance(y_pred_ha, np.ndarray):
        y_pred_ha = torch.from_numpy(y_pred_ha).float()

    ha_quantiles = (y_pred_ha, y_pred_ha, y_pred_ha)
    pb_loss = pinball_loss(quantiles, y_true, ha_quantiles).item()

    horizons_idx = [2, 5, 11]  # Corrispondono a 15m, 30m, 60m
    horizon_names = ["Step 3 (15m)", "Step 6 (30m)", "Step 12 (60m)"]
    mae_list, rmse_list = [], []

    print(f"\n=================== Baseline: Historical Average (Test Set) ===================")
    print(f"Pinball Loss (Test): {pb_loss:.4f}")
    for idx, h_name in zip(horizons_idx, horizon_names):
        yt = y_true[:, idx, :]      # (num_samples, num_sensors) -- ora seleziona il TIMESTEP giusto
        yp = y_pred_ha[:, idx, :]   # stessa correzione per la predizione HA
        mae = torch.abs(yt - yp).mean().item()
        rmse = torch.sqrt(torch.mean((yt - yp) ** 2)).item()
        mae_list.append(mae)
        rmse_list.append(rmse)
        print(f"[{h_name}] MAE: {mae:.4f} | RMSE: {rmse:.4f}")

    return pb_loss, mae_list, rmse_list


def train_and_eval_model(model, train_loader, val_loader, optimizer, scaler, quantiles, epochs, device, early_stopping, model_name='Model'):
    print(f"\n=================== Inizio Addestramento: {model_name} ===================")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for x_train, y_train in train_loader:
            x_train, y_train = x_train.to(device), y_train.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast(device_type=device.type):
                output = model(x_train)
                loss = pinball_loss(quantiles, y_train, output)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * x_train.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(device), y_val.to(device)

                with torch.amp.autocast(device_type=device.type):
                    output = model(x_val)
                    loss = pinball_loss(quantiles, y_val, output)

                val_loss += loss.item() * x_val.size(0)

            val_loss /= len(val_loader.dataset)

            early_stopping.check_early_stop(val_loss, epoch)

            if early_stopping.stop_training:
                print(f"Early stopping all'epoca {epoch}")
                break

        print(f'{model_name} | Epoca {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')


def train_tuned_model(model, train_loader, optimizer, scaler, quantiles, epochs, device, model_name='Model'):
    print(f"\n=================== Inizio Addestramento Finale: {model_name} ===================")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for x_train, y_train in train_loader:
            x_train, y_train = x_train.to(device), y_train.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast(device_type=device.type):
                output = model(x_train)
                loss = pinball_loss(quantiles, y_train, output)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * x_train.size(0)

        train_loss /= len(train_loader.dataset)
        print(f'{model_name} | Epoca {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f}')


def evaluate_model_test(model, test_loader, quantiles, device, model_name="Model"):
    """Valuta il modello addestrato sul Test Set calcolando Pinball Loss, MAE e RMSE sui 3 orizzonti."""
    model.eval()
    total_pb_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(device), y_test.to(device)
            with torch.amp.autocast(device_type=device.type):
                output = model(x_test)
                loss = pinball_loss(quantiles, y_test, output)

            total_pb_loss += loss.item() * x_test.size(0)
            all_preds.append(output[1].cpu())  # Usiamo il quantile mediano (q50) per MAE/RMSE
            all_targets.append(y_test.cpu())

    test_pb_loss = total_pb_loss / len(test_loader.dataset)
    preds_cat = torch.cat(all_preds, dim=0)
    targets_cat = torch.cat(all_targets, dim=0)

    horizons_idx = [0, 1, 2]
    horizon_names = ["Step 3 (15m)", "Step 6 (30m)", "Step 12 (60m)"]

    print(f"\n---------------- Valutazione Test Set: {model_name} ----------------")
    print(f"Pinball Loss (Test): {test_pb_loss:.4f}")

    mae_list, rmse_list = [], []
    for idx, h_name in zip(horizons_idx, horizon_names):
        yt = targets_cat[:, :, idx]
        yp = preds_cat[:, :, idx]
        mae = torch.abs(yt - yp).mean().item()
        rmse = torch.sqrt(torch.mean((yt - yp) ** 2)).item()
        mae_list.append(mae)
        rmse_list.append(rmse)
        print(f"[{h_name}] MAE: {mae:.4f} | RMSE: {rmse:.4f}")

    return test_pb_loss, mae_list, rmse_list




# Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# Preprocessing e Grafo
prep = Preprocessing(data_path='/kaggle/input/datasets/johnreds/dataset-metrla/METR-LA.csv', adj_path='/kaggle/input/datasets/johnreds/dataset-metrla/adj_METR-LA.pkl')
X_train, Y_train, X_val, Y_val, X_test, Y_test, Y_val_ha, Y_test_ha, adj_matrix, train_grouped = prep.normalization(use_graph=True)


# Matrice Laplaciana e Polinomi di Chebyshev
L_norm, A_norm = graph_to_matrices(adj_matrix)
L_norm = L_norm.to(device)
A_norm = A_norm.to(device)

# Trasformazione Tensori (Shape: B, T, N, F_in=1 e Target su orizzonti 3, 6, 12)
X_train_t = torch.from_numpy(X_train).float().unsqueeze(-1)
Y_train_t = torch.from_numpy(Y_train).float().permute(0, 2, 1)[:, :, [2, 5, 11]]

X_val_t = torch.from_numpy(X_val).float().unsqueeze(-1)
Y_val_t = torch.from_numpy(Y_val).float().permute(0, 2, 1)[:, :, [2, 5, 11]]

X_test_t = torch.from_numpy(X_test).float().unsqueeze(-1)
Y_test_t = torch.from_numpy(Y_test).float().permute(0, 2, 1)[:, :, [2, 5, 11]]

# DataLoaders (Incluso test_loader)
train_loader, val_loader, test_loader = make_loaders(
    TensorDataset(X_train_t, Y_train_t),
    TensorDataset(X_val_t, Y_val_t),
    TensorDataset(X_test_t, Y_test_t),
    batch_size=32
)

epochs = 50
quantiles = [0.1, 0.5, 0.9]

# --- 1. BASELINE HISTORICAL AVERAGE ---
ha_pb, ha_mae, ha_rmse = historical_average_baseline(Y_test, Y_test_ha, quantiles=quantiles)

# --- 2. GCN MODEL HP TUNING ---

K = [2, 3]
learning_rates = [1e-4, 1e-3]
layers = [2, 3]
prod = product( K, learning_rates, layers)
best_hp, best_model, best_val_loss = None, None, np.inf

for  k, lr, l in prod:
    print(f"\nIperparametri attuali GCN: K={k}, lr={lr}, layers={l}")
    T = chebyshev_pol(L_norm, K=k, device=device)
    model_gcn = GCN(input_dim=1, hidden_dim=32, out_dim=1, T_list=T, kernel_size=3, num_layers=l, dropout=0.3).to(device)
    optimizer_gcn = optim.Adam(model_gcn.parameters(), lr=lr)
    scaler_gcn = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')
    early_stopping = EarlyStopping(delta=0.001, verbose=True)

    train_and_eval_model(
        model_gcn, train_loader, val_loader, optimizer_gcn, scaler_gcn,
        quantiles, epochs, device, early_stopping, model_name="GCN Model Tuning"
    )
    if early_stopping.best_loss < best_val_loss:
        best_val_loss = early_stopping.best_loss
        best_hp = (k, lr, l)
        best_epochs = early_stopping.best_epoch


(k, lr, l) = best_hp
print(f"\nMigliori Iperparametri GCN: K={k}, lr={lr}, layers={l} | Epoche: {best_epochs}")

# Addestramento Finale e Valutazione Test GCN
T = chebyshev_pol(L_norm, K=k, device=device)
model_gcn = GCN(input_dim=1, hidden_dim=32, out_dim=1, T_list=T, kernel_size=3, num_layers=l, dropout=0.3).to(device)
optimizer_gcn = optim.Adam(model_gcn.parameters(), lr=lr)
scaler_gcn = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

train_tuned_model(model_gcn, train_loader, optimizer_gcn, scaler_gcn, quantiles, best_epochs, device, model_name="GCN Tuned")
gcn_pb, gcn_mae, gcn_rmse = evaluate_model_test(model_gcn, test_loader, quantiles, device, model_name="GCN Model (Spectral)")

# --- 3. TEMPORAL MODEL (NO GRAPH) ---
model_temporal = Temporal_Model(input_dim=1, out_dim=3, hidden_dim=32, kernel_size=3, num_layers=l, dropout=0.3).to(device)
optimizer_temporal = optim.Adam(model_temporal.parameters(), lr=lr)
scaler_temporal = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')
early_stopping_temp = EarlyStopping(delta=0.001, verbose=True)

train_and_eval_model(
    model_temporal, train_loader, val_loader, optimizer_temporal, scaler_temporal,
    quantiles, epochs, device, early_stopping_temp, model_name="Temporal Model (No Graph)"
)
temp_pb, temp_mae, temp_rmse = evaluate_model_test(model_temporal, test_loader, quantiles, device, model_name="Temporal Model (No Graph)")

# --- 4. GIN MODEL (SPATIAL GRAPH) ---
eps_candidates = [1e-4, 1e-3, 1e-2]

best_gin_eps = None
best_gin_val_loss = np.inf
best_gin_epochs = 0

for eps in eps_candidates:
    model_gin = GIN(
        input_dim=1, out_dim=1, hidden_dim=32, A=A_norm,
        kernel_size=3, num_layers=l, dropout=0.3, eps=eps
    ).to(device)

    optimizer_gin = optim.Adam(model_gin.parameters(), lr=lr)
    scaler_gin = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')
    early_stopping_gin = EarlyStopping(delta=0.001, verbose=True)

    train_and_eval_model(
        model_gin, train_loader, val_loader, optimizer_gin, scaler_gin,
        quantiles, epochs, device, early_stopping_gin, model_name=f"GIN Model (eps={eps})"
    )

    if early_stopping_gin.best_loss < best_gin_val_loss:
        best_gin_val_loss = early_stopping_gin.best_loss
        best_gin_eps = eps
        best_gin_epochs = early_stopping_gin.best_epoch

# Addestramento Finale e Valutazione Test GIN
model_gin = GIN(
    input_dim=1, out_dim=1, hidden_dim=32, A=A_norm,
    kernel_size=3, num_layers=l, dropout=0.3, eps=best_gin_eps
).to(device)

optimizer_gin = optim.Adam(model_gin.parameters(), lr=lr)
scaler_gin = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

train_tuned_model(model_gin, train_loader, optimizer_gin, scaler_gin, quantiles, best_gin_epochs, device, model_name="GIN Tuned")
gin_pb, gin_mae, gin_rmse = evaluate_model_test(model_gin, test_loader, quantiles, device, model_name="GIN Model (Spatial)")

# --- TABELLA RIASSUNTIVA FINALE SUL TEST SET ---
print("\n" + "="*80)
print("                       RIEPILOGO METRICHE TEST SET")
print("="*80)
print(f"{'Modello':<25} | {'Pinball Loss':<12} | {'MAE (15m/30m/60m)':<22} | {'RMSE (15m/30m/60m)'}")
print("-" * 80)
print(f"{'Historical Average':<25} | {ha_pb:<12.4f} | {f'{ha_mae[0]:.2f}/{ha_mae[1]:.2f}/{ha_mae[2]:.2f}':<22} | {f'{ha_rmse[0]:.2f}/{ha_rmse[1]:.2f}/{ha_rmse[2]:.2f}'}")
print(f"{'Temporal (No Graph)':<25} | {temp_pb:<12.4f} | {f'{temp_mae[0]:.2f}/{temp_mae[1]:.2f}/{temp_mae[2]:.2f}':<22} | {f'{temp_rmse[0]:.2f}/{temp_rmse[1]:.2f}/{temp_rmse[2]:.2f}'}")
print(f"{'GCN (Spectral)':<25} | {gcn_pb:<12.4f} | {f'{gcn_mae[0]:.2f}/{gcn_mae[1]:.2f}/{gcn_mae[2]:.2f}':<22} | {f'{gcn_rmse[0]:.2f}/{gcn_rmse[1]:.2f}/{gcn_rmse[2]:.2f}'}")
print(f"{'GIN (Spatial)':<25} | {gin_pb:<12.4f} | {f'{gin_mae[0]:.2f}/{gin_mae[1]:.2f}/{gin_mae[2]:.2f}':<22} | {f'{gin_rmse[0]:.2f}/{gin_rmse[1]:.2f}/{gin_rmse[2]:.2f}'}")
print("="*80)

import os


# Directory di output garantita da Kaggle
save_dir = "/kaggle/working"
os.makedirs(save_dir, exist_ok=True)

# 1. Definizione dei percorsi espliciti
path_gcn = os.path.join(save_dir, "gcn_final_complete.pt")
path_temporal = os.path.join(save_dir, "temporal_final.pt")
path_gin = os.path.join(save_dir, "gin_final_complete.pt")

# 2. Salvataggio Pesi e Iperparametri
torch.save(
    {
        "state_dict": model_gcn.state_dict(),
        "hyperparameters": {"K": k, "num_layers": l, "lr": lr},
    },
    path_gcn,
)

torch.save(model_temporal.state_dict(), path_temporal)

torch.save(
    {
        "state_dict": model_gin.state_dict(),
        "hyperparameters": {"eps": best_gin_eps, "num_layers": l, "lr": lr},
    },
    path_gin,
)

# 3. Verifica immediata per i log di Kaggle
print("\n--- STATO DEI FILE SALVATI IN /kaggle/working/ ---")
for file_path in [path_gcn, path_temporal, path_gin]:
  if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"Disponibile al download: {file_path} ({size_mb:.2f} MB)")
  else:
    print(f"ERRORE: {file_path} non è stato scritto su disco!")

cuda
Numero di NaN residui: 0
Global Mean (Train): 58.6335
Global Std (Train):  12.6469

=================== Baseline: Historical Average (Test Set) ===================
Pinball Loss (Test): 0.4369
[Step 3 (15m)] MAE: 0.2913 | RMSE: 0.5827
[Step 6 (30m)] MAE: 0.2913 | RMSE: 0.5827
[Step 12 (60m)] MAE: 0.2913 | RMSE: 0.5827

Iperparametri attuali GCN: K=2, lr=0.0001, layers=2

=================== Inizio Addestramento: GCN Model Tuning ===================
GCN Model Tuning | Epoca 01/50 | Train Loss: 0.7021 | Val Loss: 0.6341
GCN Model Tuning | Epoca 02/50 | Train Loss: 0.6116 | Val Loss: 0.5870
GCN Model Tuning | Epoca 03/50 | Train Loss: 0.5680 | Val Loss: 0.5496
GCN Model Tuning | Epoca 04/50 | Train Loss: 0.5263 | Val Loss: 0.5194
GCN Model Tuning | Epoca 05/50 | Train Loss: 0.4953 | Val Loss: 0.4898
GCN Model Tuning | Epoca 06/50 | Train Loss: 0.4728 | Val Loss: 0.4767
GCN Model Tuning | Epoca 07/50 | Train Loss: 0.4538 | Val Loss: 0.4617
GCN Model Tuning | Epoca 08/50 | Train Loss: 0